In [1]:
import torch
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, Module
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [2]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).float()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [3]:
class CustomModel(Module):
    def __init__(self, input_size: int, output_size) -> None:
        super().__init__()
        self.net = Sequential(
            Linear(input_size, 32),
            ReLU(),
            Linear(32,16),
            ReLU(),
            Linear(16, output_size)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [4]:
def train_model_reg(features, targets, kf, batch, epochs, criterion):
    r2_scores = []
    mse_scores = []
    rmse_scores = []
    for i, (train_index, test_index) in enumerate(kf.split(features)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)
        model = CustomModel(10, 1)
        optmizer = Adam(model.parameters(), lr=0.001)
        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]
            result = model(x_test_tensor).cpu().numpy()
            r2_s = r2_score(y_true, result)
            rmse_s = root_mean_squared_error(y_true, result)
            mse_s = mean_squared_error(y_true, result)
            r2_scores.append(r2_s)
            rmse_scores.append(rmse_s)
            mse_scores.append(mse_s)
            print(f"R^2: {r2_s}")
            print(f"RMSE: {rmse_s}")
            print(f"MSE: {mse_s}")

    print(f"\nResultado modelo ----- R^2: {np.mean(r2_scores)}, RMSE: {np.mean(rmse_scores)}, MSE: {np.mean(mse_scores)}")

In [5]:
dataset = load_diabetes()
features = dataset.data
targets = np.expand_dims(dataset.target, axis=1)

In [6]:
criterion = MSELoss()
max_epochs = 500
batch = 12
kf = KFold(n_splits=5, shuffle=True)

In [7]:
train_model_reg(features, targets, kf, batch, max_epochs, criterion)

Treinando modelo ----- Fold 1
R^2: 0.5557900439924308
RMSE: 46.53351748922935
MSE: 2165.3682499204137
Treinando modelo ----- Fold 2
R^2: 0.48859399477339827
RMSE: 56.804620437048804
MSE: 3226.764902997183
Treinando modelo ----- Fold 3
R^2: 0.46585133468113793
RMSE: 52.17007764314843
MSE: 2721.717001292136
Treinando modelo ----- Fold 4
R^2: 0.5176601452581622
RMSE: 57.83450533880557
MSE: 3344.83000778433
Treinando modelo ----- Fold 5
R^2: 0.42372841750179957
RMSE: 60.00704033426128
MSE: 3600.8448896776604

Resultado modelo ----- R^2: 0.4903247872413858, RMSE: 54.66995224849868, MSE: 3011.9050103343448
